## Import

In [1]:
import warnings
warnings.filterwarnings("ignore")

import os
import time
import numpy as np
import scipy.linalg as linalg
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
#import preprocess


device = 'cuda:1'

%load_ext autoreload
%autoreload 2

## Data generation

In [2]:
from datasets.Uniform import Uniform

n, d = 10000, 64                                 # < higher d, higher MI
eps = 0.225                                      # < lower eps, higher MI


dataset = Uniform(n_samples=n, n_dims=d//2, eps=eps)
X, Y = dataset.sample_data(n)
X, Y = X.to(device), Y.to(device)
MI = dataset.true_mutual_info()


print('X size=', X.size(), 'Y size=', Y.size())
print("True MI is", MI)

X size= torch.Size([10000, 32]) Y size= torch.Size([10000, 32])
True MI is 32.75224627896869


## MI estimation

In [3]:
class Hyperparams(object):
    def __init__(self): 
        self.critic = 'neural'                # ('neural', 'quadratic')
        self.lr = 5e-4
        self.bs = 500
        self.n_bridges = 4
        self.wd = 1e-5
        self.max_iteration = 1250
        
hyperparams=Hyperparams()

architecture_critic = [d, 500, 500, 500, 1]

In [5]:
## Mutual information neural diffusion estimate (MINDE)
from estimators.MINDE import MINDE

hyperparams.t_patience = 500
hyperparams.dim = d//2
hyperparams.device = device
hyperparams.importance_sampling = True

estimator = MINDE(None, None, None, hyperparams)

estimator.to(device)
estimator.learn(X, Y)

print('true MI:', MI)
print('est MI:', estimator.MI(X, Y))

use ema: True bs: 500
finished: t= 0 loss= 1.0042386054992676 loss val= 0.9974202513694763 best val loss= 0.9974202513694763 best t= 0
finished: t= 63 loss= 0.30585378408432007 loss val= 0.3531750738620758 best val loss= 0.3093861937522888 best t= 60
finished: t= 126 loss= 0.3219917118549347 loss val= 0.35831013321876526 best val loss= 0.29018229246139526 best t= 125
finished: t= 189 loss= 0.4154229462146759 loss val= 0.35713765025138855 best val loss= 0.29018229246139526 best t= 125
finished: t= 252 loss= 0.3588435649871826 loss val= 0.3575206995010376 best val loss= 0.2869532108306885 best t= 232
finished: t= 315 loss= 0.35504236817359924 loss val= 0.30824151635169983 best val loss= 0.2869532108306885 best t= 232
finished: t= 378 loss= 0.2912098467350006 loss val= 0.3950827717781067 best val loss= 0.2821611762046814 best t= 366
finished: t= 441 loss= 0.39036571979522705 loss val= 0.4146738648414612 best val loss= 0.2821611762046814 best t= 366
finished: t= 504 loss= 0.354185760021209

In [6]:
## Mutual information neural estimate (MINE)
from estimators.MINE import MINE

estimator = MINE(None, None, architecture_critic, hyperparams)
estimator.to(device)
estimator.learn(X, Y)


print('true MI:', MI)
print('est MI:', estimator.MI(X, Y))

finished: t= 0 loss= 0.0011338740587234497 loss val= 0.0015091821551322937 best val loss= 0.0015091821551322937 best t= 0
finished: t= 63 loss= -5.462620735168457 loss val= -6.452919960021973 best val loss= -6.661571502685547 best t= 59
finished: t= 126 loss= -6.827361106872559 loss val= -6.711553573608398 best val loss= -6.947531700134277 best t= 124
finished: t= 189 loss= -5.4113054275512695 loss val= -6.702811241149902 best val loss= -6.947531700134277 best t= 124
finished: t= 252 loss= -5.944204330444336 loss val= -6.374396324157715 best val loss= -7.088308334350586 best t= 213
finished: t= 315 loss= -6.029181480407715 loss val= -6.834135055541992 best val loss= -7.171073913574219 best t= 271
finished: t= 378 loss= -5.470160007476807 loss val= -6.920225143432617 best val loss= -7.171073913574219 best t= 271
finished: t= 441 loss= -6.481607913970947 loss val= -6.80627965927124 best val loss= -7.226330757141113 best t= 379
finished: t= 504 loss= -6.048641204833984 loss val= -6.452892

In [4]:
## Neural adaptive MI estimate
from estimators.VCE import VCE

hyperparams.K = 8

estimator = VCE(None, None, architecture_critic, hyperparams)
estimator.to(device)
estimator.learn(X, Y)

print('true MI:', MI)
print('est MI:', estimator.MI(X, Y))

K components= 5 copula transform= True
nde type: FM
finished: t= 0 loss= 1.331752896308899 loss val= 1.3457238674163818 best val loss= 1.3457238674163818 best t= 0
finished: t= 126 loss= 0.4343101978302002 loss val= 0.467659592628479 best val loss= 0.4576948285102844 best t= 112
finished: t= 252 loss= 0.4679527282714844 loss val= 0.46991294622421265 best val loss= 0.4537099301815033 best t= 156
finished: t= 378 loss= 0.4688258171081543 loss val= 0.4652165472507477 best val loss= 0.4535630941390991 best t= 299
finished: t= 504 loss= 0.46807634830474854 loss val= 0.47278472781181335 best val loss= 0.44476401805877686 best t= 439
finished: t= 630 loss= 0.5031611323356628 loss val= 0.47838541865348816 best val loss= 0.44476401805877686 best t= 439


finished: t= 0 loss= 1.3435962200164795 loss val= 1.3475816249847412 best val loss= 1.3475816249847412 best t= 0
finished: t= 126 loss= 0.48100659251213074 loss val= 0.5113951563835144 best val loss= 0.4931228756904602 best t= 66
finished: t= 2

In [7]:
## Neural adaptive MI estimate
from estimators.fDIME import fDIME

estimator = fDIME(None, None, architecture_critic, hyperparams)
estimator.to(device)
estimator.learn(X, Y)

print('true MI:', MI)
print('est MI:', estimator.MI(X, Y))

finished: t= 0 loss= 1.3869600296020508 loss val= 1.3868474960327148 best val loss= 1.3868474960327148 best t= 0
finished: t= 63 loss= 27.631023406982422 loss val= 27.631023406982422 best val loss= 0.0034329858608543873 best t= 28
finished: t= 126 loss= 27.631023406982422 loss val= 27.631023406982422 best val loss= 0.0034329858608543873 best t= 28
finished: t= 189 loss= 27.631023406982422 loss val= 27.631023406982422 best val loss= 0.0034329858608543873 best t= 28


true MI: 32.75224627896869
est MI: 7.694808483123779
